# SpixRWKV-7: Full Architecture Benchmark on Tiny-ImageNet

Trains all 6 architecture variants (spix, conv, gnn, vq, hybrid, vit) on Tiny-ImageNet with:
- **2x T4 GPUs** via DataParallel + mixed precision (AMP)
- **Early stopping** with patience=5 on validation accuracy
- **~10 hour budget** — estimated ~15 min per tiny model, ~1.5 hours per small model

## Setup

In [ ]:
# Install spixrwkv7 package
!pip install -q git+https://github.com/Gabz4200/SpixRWKV7.git#egg=spixrwkv7[all]

# Verify GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
import os
import time
import json
import math
import random
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from PIL import Image

from spixrwkv7 import (
    create_vision_rwkv7,
    create_optimized_vision_rwkv7,
    create_conv_vision_rwkv7,
    create_gnn_vision,
    create_vq_rwkv7,
    create_hybrid_vision,
    ClassificationHead,
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Imports OK")

## Configuration

In [ ]:
# ── Training config ──
IMG_SIZE = 64  # Tiny-ImageNet native resolution
NUM_CLASSES = 200
MAX_EPOCHS = 20
EARLY_STOP_PATIENCE = 5
BATCH_SIZE = 64  # Per GPU; effective = 64 * num_gpus
LR = 3e-4
WEIGHT_DECAY = 0.05
GRAD_CLIP = 1.0
NUM_WORKERS = 2

# ── Device ──
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
print(f"Device: {DEVICE}, GPUs: {NUM_GPUS}")

# ── Model configs (tiny scale) ──
# Each tuple: (model_type, builder_name, config_override)
MODELS = [
    ("vit",     None,              {}),
    ("gnn",     "gnn",             {"depth": 3, "gnn_conv": "gatv2", "gnn_heads": 4}),
    ("conv",    "conv",            {"depth": 3, "conv_stem_norm": "layernorm"}),
    ("spix",    "spix",            {"depth": 3}),
    ("hybrid",  "hybrid",          {"depth": 4, "num_rwkv_layers": 1, "num_gnn_layers": 1}),
    ("vq",      "vq",              {"depth": 2, "codebook_size": 128}),
]

# Shared backbone config
BACKBONE_CFG = {
    "embed_dims": 128,
    "num_heads": 2,
    "num_superpixels": 36,
    "diff_slic_iters": 5,
    "compactness": 0.5,
    "drop_path_rate": 0.1,
    "init_values": 1e-5,
    "norm_layer": "rmsnorm",
    "act_layer": "swiglu",
}

print(f"Training config: {MAX_EPOCHS} epochs, batch_size={BATCH_SIZE}, lr={LR}")
print(f"Models to train: {[m[0] for m in MODELS]}")

## Dataset: Tiny-ImageNet

In [ ]:
class TinyImageNet(Dataset):
    """Tiny-ImageNet from HuggingFace: 100K train, 10K val, 200 classes, 64x64."""

    def __init__(self, split: str, img_size: int = 64):
        from datasets import load_dataset
        self.ds = load_dataset("zh-plus/tiny-imagenet", split=split)
        self.img_size = img_size
        # Pre-compute OkLAB conversion
        from spixrwkv7.data.colors import from_linear_rgb_to_oklab
        self._to_oklab = from_linear_rgb_to_oklab

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        img = item["image"]
        label = item["label"]

        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.array(img))
        if img.mode != "RGB":
            img = img.convert("RGB")

        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        img_np = torch.tensor(np.array(img), dtype=torch.uint8)
        img_float = img_np.permute(2, 0, 1).float() / 255.0

        # sRGB -> Linear RGB -> OkLAB
        srgb = img_float.unsqueeze(0)
        linear_rgb = srgb.clamp(0).pow(2.2)
        oklab = self._to_oklab(linear_rgb).squeeze(0)

        # Alpha + spatial coordinates
        alpha = torch.ones(1, self.img_size, self.img_size)
        yy, xx = torch.meshgrid(
            torch.linspace(-1, 1, self.img_size),
            torch.linspace(-1, 1, self.img_size),
            indexing="ij",
        )
        xy = torch.stack([xx, yy], dim=0)

        x = torch.cat([oklab, alpha, xy], dim=0)  # (6, 64, 64)
        return x, label


# Load datasets
print("Loading Tiny-ImageNet...")
train_ds = TinyImageNet("train", IMG_SIZE)
val_ds = TinyImageNet("valid", IMG_SIZE)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Classes: {NUM_CLASSES}")

# DataLoaders
pin = DEVICE.type == "cuda"
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=pin, drop_last=pin,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin,
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## Model Builders

In [ ]:
def build_model(model_type: str, overrides: dict) -> Tuple[nn.Module, nn.Module]:
    """Build backbone + head for a given model type."""
    cfg = {**BACKBONE_CFG, **overrides}

    if model_type == "vit":
        # Simple ViT baseline
        backbone = SimpleViT(
            img_size=IMG_SIZE, patch_size=8, in_chans=6,
            num_classes=NUM_CLASSES, embed_dim=cfg["embed_dims"],
            depth=6, num_heads=cfg["num_heads"],
        )
        return backbone, None

    builder_map = {
        "spix": create_optimized_vision_rwkv7,
        "conv": create_conv_vision_rwkv7,
        "gnn": create_gnn_vision,
        "vq": create_vq_rwkv7,
        "hybrid": create_hybrid_vision,
    }
    builder = builder_map[model_type]

    # Common kwargs
    kwargs = {
        "img_size": IMG_SIZE,
        "embed_dims": cfg["embed_dims"],
        "num_heads": cfg["num_heads"],
        "depth": cfg["depth"],
        "init_values": cfg["init_values"],
        "final_norm": True,
        "out_indices": [-1],
        "norm_layer": cfg["norm_layer"],
        "act_layer": cfg["act_layer"],
        "scatter_output": model_type != "vq",
    }

    # Model-specific kwargs
    if model_type in ("spix", "conv", "gnn", "hybrid"):
        kwargs.update({
            "num_superpixels": cfg["num_superpixels"],
            "diff_slic_iters": cfg["diff_slic_iters"],
            "compactness": cfg["compactness"],
            "drop_path_rate": cfg["drop_path_rate"],
        })
    if model_type in ("spix", "conv"):
        kwargs["spixel_backend"] = "diff_slic"
    if model_type == "conv":
        kwargs["conv_stem_norm"] = cfg.get("conv_stem_norm", "layernorm")
        kwargs["conv_post_norm"] = "layernorm"
    if model_type == "gnn":
        kwargs["gnn_conv"] = cfg.get("gnn_conv", "gatv2")
        kwargs["gnn_heads"] = cfg.get("gnn_heads", 4)
        kwargs["gnn_aggr"] = "mean"
    if model_type == "hybrid":
        kwargs["num_rwkv_layers"] = cfg.get("num_rwkv_layers", 1)
        kwargs["num_gnn_layers"] = cfg.get("num_gnn_layers", 1)
        kwargs["gnn_conv"] = "gatv2"
        kwargs["gnn_heads"] = 4
        kwargs["register_tokens"] = 4
    if model_type == "vq":
        kwargs["codebook_size"] = cfg.get("codebook_size", 128)
        kwargs["downsample_factor"] = 8

    backbone = builder(**kwargs)
    head = ClassificationHead(cfg["embed_dims"], NUM_CLASSES)
    return backbone, head


class SimpleViT(nn.Module):
    """ViT baseline with 6-channel input."""
    def __init__(self, img_size, patch_size, in_chans, num_classes,
                 embed_dim, depth, num_heads):
        super().__init__()
        self.patch_embed = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        num_patches = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + num_patches, embed_dim))
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads,
                                       dim_feedforward=4 * embed_dim, batch_first=True)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        nn.init.xavier_uniform_(self.cls_token)
        nn.init.xavier_uniform_(self.pos_embed)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x[:, 0])


print("Model builders OK")

## Training Loop with Early Stopping

In [ ]:
def train_model(model_type: str, overrides: dict) -> dict:
    """Train one model variant with early stopping. Returns results dict."""
    print(f"\n{'='*70}")
    print(f"  Training: {model_type.upper()}")
    print(f"{'='*70}")

    # Build model
    backbone, head = build_model(model_type, overrides)
    backbone = backbone.to(DEVICE)
    if head is not None:
        head = head.to(DEVICE)

    # Multi-GPU
    if NUM_GPUS > 1:
        backbone = nn.DataParallel(backbone)
        if head is not None:
            head = nn.DataParallel(head)
        print(f"  Using {NUM_GPUS} GPUs via DataParallel")

    # AMP
    use_amp = DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    # Params
    params = list(backbone.parameters())
    if head is not None:
        params += list(head.parameters())
    total_params = sum(p.numel() for p in params)
    print(f"  Params: {total_params / 1e6:.2f}M")

    # Optimizer
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=MAX_EPOCHS, eta_min=1e-6
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # Early stopping
    best_val_acc = 0.0
    best_epoch = 0
    patience_counter = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}

    total_start = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()

        # ── Train ──
        backbone.train()
        if head is not None:
            head.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch_idx, (x, y) in enumerate(train_loader):
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=use_amp):
                outs = backbone(x)
                feat = outs[0] if isinstance(outs, tuple) else outs
                if head is not None:
                    logits = head(feat) if not isinstance(head, nn.DataParallel) else head(feat)
                else:
                    logits = feat
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, max_norm=GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * x.size(0)
            train_correct += (logits.argmax(1) == y).sum().item()
            train_total += x.size(0)

        scheduler.step()

        train_loss /= train_total
        train_acc = 100.0 * train_correct / train_total

        # ── Validate ──
        backbone.eval()
        if head is not None:
            head.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    outs = backbone(x)
                    feat = outs[0] if isinstance(outs, tuple) else outs
                    if head is not None:
                        logits = head(feat)
                    else:
                        logits = feat
                    loss = criterion(logits, y)

                val_loss += loss.item() * x.size(0)
                val_correct += (logits.argmax(1) == y).sum().item()
                val_total += x.size(0)

        val_loss /= val_total
        val_acc = 100.0 * val_correct / val_total
        gap = train_acc - val_acc
        lr_now = optimizer.param_groups[0]["lr"]
        epoch_time = time.time() - epoch_start

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(lr_now)

        marker = ""
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            patience_counter = 0
            marker = " *best*"
        else:
            patience_counter += 1

        print(
            f"  Epoch {epoch:>2}/{MAX_EPOCHS} | "
            f"train_loss={train_loss:.4f} acc={train_acc:.1f}% | "
            f"val_loss={val_loss:.4f} acc={val_acc:.1f}% | "
            f"gap={gap:.1f}% | {epoch_time:.0f}s{marker}"
        )

        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"  Early stopping at epoch {epoch} (patience={EARLY_STOP_PATIENCE})")
            break

    total_time = time.time() - total_start

    # Cleanup GPU memory
    del backbone, head
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    result = {
        "model_type": model_type,
        "total_params": total_params,
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
        "final_train_acc": history["train_acc"][-1],
        "final_val_acc": history["val_acc"][-1],
        "final_gap": history["train_acc"][-1] - history["val_acc"][-1],
        "epochs_run": len(history["train_acc"]),
        "total_time_s": total_time,
        "history": history,
    }

    print(f"  RESULT: best_val={best_val_acc:.2f}% (epoch {best_epoch}), time={total_time:.0f}s")
    return result


print("Training loop OK")

## Run All Models

In [ ]:
all_results = []
total_start = time.time()
TIME_BUDGET_S = 10 * 3600  # 10 hours

for model_type, builder_key, overrides in MODELS:
    elapsed = time.time() - total_start
    if elapsed > TIME_BUDGET_S * 0.9:
        print(f"\n  Time budget nearly exhausted ({elapsed/3600:.1f}h / 10h). Skipping remaining models.")
        break

    remaining_h = (TIME_BUDGET_S - elapsed) / 3600
    print(f"\n  Time remaining: {remaining_h:.1f}h")

    try:
        result = train_model(model_type, overrides)
        all_results.append(result)
    except Exception as e:
        print(f"  ERROR training {model_type}: {e}")
        import traceback
        traceback.print_exc()
        all_results.append({"model_type": model_type, "error": str(e)})

total_time = time.time() - total_start
print(f"\nTotal training time: {total_time/3600:.2f}h")

## Results Summary

In [ ]:
print("=" * 80)
print("  FINAL RESULTS — Tiny-ImageNet Generalization Benchmark")
print("=" * 80)
print(f"  {'Model':<10} {'Params':>10} {'Best Val':>10} {'Final Val':>10} {'Gap':>8} {'Epochs':>8} {'Time':>8}")
print(f"  {'-' * 70}")

for r in all_results:
    if "error" in r:
        print(f"  {r['model_type']:<10} ERROR: {r['error']}")
        continue
    print(
        f"  {r['model_type']:<10} {r['total_params']/1e6:>9.2f}M "
        f"{r['best_val_acc']:>9.2f}% {r['final_val_acc']:>9.2f}% "
        f"{r['final_gap']:>7.1f}% {r['epochs_run']:>7} {r['total_time_s']:>7.0f}s"
    )

print(f"\n  Total time: {total_time/3600:.2f}h")
print("=" * 80)

## Save Results

In [ ]:
# Save results to JSON
output_path = Path("/kaggle/working/results_tiny_imagenet.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

serializable = []
for r in all_results:
    d = {k: v for k, v in r.items() if k != "history"}
    serializable.append(d)

with open(output_path, "w") as f:
    json.dump({
        "config": {
            "img_size": IMG_SIZE,
            "num_classes": NUM_CLASSES,
            "max_epochs": MAX_EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "seed": SEED,
            "device": str(DEVICE),
            "num_gpus": NUM_GPUS,
        },
        "results": serializable,
    }, f, indent=2)

print(f"Results saved to {output_path}")

## Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for r in all_results:
    if "error" in r or "history" not in r:
        continue
    h = r["history"]
    epochs = list(range(1, len(h["train_acc"]) + 1))

    axes[0].plot(epochs, h["train_acc"], "--", alpha=0.7, label=f"{r['model_type']} train")
    axes[0].plot(epochs, h["val_acc"], "-", alpha=0.9, label=f"{r['model_type']} val")
    axes[0].set_title("Accuracy (%)")
    axes[0].set_xlabel("Epoch")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, h["train_loss"], "--", alpha=0.7, label=f"{r['model_type']} train")
    axes[1].plot(epochs, h["val_loss"], "-", alpha=0.9, label=f"{r['model_type']} val")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    gaps = [h["train_acc"][i] - h["val_acc"][i] for i in range(len(h["train_acc"]))]
    axes[2].plot(epochs, gaps, "-", alpha=0.9, label=r["model_type"])
    axes[2].set_title("Overfitting Gap (train - val)")
    axes[2].set_xlabel("Epoch")
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3)

plt.suptitle("SpixRWKV-7 Architecture Comparison on Tiny-ImageNet", fontsize=14)
plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved training_curves.png")

## Architecture Analysis

In [ ]:
successful = [r for r in all_results if "error" not in r]
if successful:
    best = max(successful, key=lambda r: r["best_val_acc"])
    most_efficient = min(successful, key=lambda r: r["total_params"])
    fastest = min(successful, key=lambda r: r["total_time_s"])
    least_overfit = min(successful, key=lambda r: r["final_gap"])

    print("=" * 70)
    print("  ANALYSIS")
    print("=" * 70)
    print(f"  Best accuracy:      {best['model_type']} ({best['best_val_acc']:.2f}%)")
    print(f"  Most parameter-efficient: {most_efficient['model_type']} ({most_efficient['total_params']/1e6:.2f}M)")
    print(f"  Fastest to train:   {fastest['model_type']} ({fastest['total_time_s']:.0f}s)")
    print(f"  Least overfitting:  {least_overfit['model_type']} (gap={least_overfit['final_gap']:.1f}%)")
    print("=" * 70)
else:
    print("No successful runs to analyze.")